# Step 1: Parameters & Config - All Markets

In [0]:
# PARAMETERS — ALL MARKETS

markets_to_process = [
    # {'Local_Market': 'Germany','min_period': '2023-01-08', 'max_period': '2026-07-26'},
    {'Local_Market': 'Italy', 'min_period': '2023-04-09', 'max_period': '2026-06-28'},
    # {'Local_Market': 'UK','min_period': '2023-06-10', 'max_period': '2026-07-25'},
]

# Column mappings per market — maps Circana file column headers to standard names.
# manufacturer_col: column in the Circana Excel/CSV containing manufacturer names
# value_col / volume_col: sales measures (currency & units) — names vary by language/market
# mat_database: the Database key used to look up MAT flags in period_mat_ytd
market_config = {
    'Germany': {'manufacturer_col': 'HERSTELLER_TWH1 [ HERSTELLER_TWH1 ]','value_col':'Verkauf Euro','volume_col': 'Verkauf Menge','mat_database': 'DE_R_NE3986'},
    'Italy': {'manufacturer_col': 'Produttore Valori','value_col':'Vendite in Valore', 'volume_col':'Vendite in Volume','mat_database': 'NE_NESTLEBARRETTE_IT'},
    'UK': {'manufacturer_col': 'MANUFACTURER_MBA1 [ MANUFACTURER_MBA1 ]','value_col':'Value Sales', 'volume_col':'Volume Sales','mat_database': 'Cereal Bars'}
}

# Market-specific category/sub-category filters applied to both ADB and Discover queries.
# Local_category_col: top-level category filter (always 'Bars' for Bars Category)
# Local_Sub_Category_col: sub-categories to include in the ADB WHERE clause
Filters_config = {
    'Germany': {'Local_category_col': 'Bars','Local_Sub_Category_col': ['CEREALIENRIEGEL', 'FLAPJACKS', 'FRUCHTSCHNITTEN', 'FUNKTIONALE RIEGEL', 'MUESLIRIEGEL', 'MUESLI-KEKS/-KONFEKT', 'SANDWICH-RIEGEL', 'ALL OTHERS']},
    'Italy': {'Local_category_col': 'Bars','Local_Sub_Category_col': ['ADULTO','BAMBINO']},
    'UK': {'Local_category_col': 'Bars'},
}

# ============================================================
# MARKET-WISE EXCEPTIONS CONFIG
# Consolidates all market-specific processing logic into one dict.
# Each key controls a different aspect of the comparison pipeline:
#   period_format      : 'date' (yyyy-MM-dd) or 'week' (YYYY_W##) for period grouping
#   add_mat_flag       : whether to join MAT flags (TY/LY/2LY) onto ADB data
#   product_exclusion  : SQL fragment to exclude specific product_ids from ADB
#   mfr_comparison_type: how manufacturer totals are compared (mat, mat_join, category_level, all_periods)
#   period_comparison_type: how period totals are compared (mat_flags, category_level, all_periods)
#   mfr_name_transforms: list of rename/strip rules to align Circana vs ADB manufacturer names
#   disc_min_period_override: override Discover start date (e.g. when ADB data starts later)
#   circana_file       : explicit Circana filename
#   circana_sheet      : specific sheet within the Circana file to read
# ============================================================
exceptions_config = {
    'Germany': {
        'period_format': 'week',           # YYYY_W## (ISO week-level grouping)
        'add_mat_flag': True,
        'product_exclusion': None,  # No product exclusion for Germany
        'mfr_col': 'p.Local_Manufacturer',  # Use LOCAL manufacturer names to match Circana's HERSTELLER_TWH1
        'mfr_comparison_type': 'all_periods',  # Aggregate all periods (no MAT filter)
        'period_comparison_type': 'all_periods',  # Period comparison also uses all_periods
        'mfr_name_transforms': None,
        'disc_min_period_override': '2023-01-08',  # ISO W1 2023 — aligned with min_period
        'circana_file': None,
        'circana_sheet': None,
    },
    'Italy': {
        'period_format': 'date',
        'add_mat_flag': True,
        'product_exclusion': None,
        'mfr_col': 'p.Local_Manufacturer',  # Use Italian manufacturer names to match Circana
        'mfr_comparison_type': 'all_periods',  # Aggregate all periods (no MAT filter) — matches reference pivot
        'period_comparison_type': 'mat_flags',  # Period comparison also uses MAT flags
        'mfr_name_transforms': None,
        'disc_min_period_override': None,
        'circana_file': None,
        'circana_sheet': None,
    },
    # UK: No MAT flag join; ADB queried at category level (no sub-category filter);
    # Manufacturer names need prefix stripping and renaming to align with Circana;
    # Uses explicit Circana file with 'Unify File' sheet for product-level data
    'UK': {
        'period_format': 'date',
        'add_mat_flag': False,             # MAT_Flag NOT added to adb_local
        'product_exclusion': "AND p.product_id != 'H00190058@H01_L009'",  # Specific product exclusion
        'mfr_comparison_type': 'category_level',  # ADB at category level, no sub-category
        'period_comparison_type': 'category_level',  # ADB at category level for periods too
        'mfr_name_transforms': [
            {'type': 'strip_prefix', 'pattern': r'^CEREALS_'},         # Strip CEREALS_ prefix
            {'type': 'rename', 'from': 'NATURES PATH', 'to': 'NATURES PATH FOODS'},  # Align naming
        ],
        'disc_min_period_override': None,
        'circana_file': None,    # Auto-detect via glob
        'circana_sheet': None,               # Sheet within the file
    },
}

# MAT flags used across comparison steps (Moving Annual Total: This Year, Last Year, 2 Years Ago)
_mat_flags = ['MAT TY', 'MAT LY', 'MAT 2LY']

print(f"Markets to process: {[m['Local_Market'] for m in markets_to_process]}")
print(f"ADB Schema: conformed")
print(f"Exceptions configured for: {list(exceptions_config.keys())}")

Markets to process: ['Germany']
ADB Schema: conformed
Exceptions configured for: ['Germany', 'Italy', 'UK']


# Step 2: Conformed — Query ADB for All Markets

In [0]:
# --- Imports ---
from pyspark.sql.functions import col, date_format, sum as spark_sum, round as spark_round, expr, weekofyear, lpad, concat, lit, coalesce, to_date, regexp_replace, when
import os
import pandas as pd

# Store per-market results for downstream comparison steps ---
adb_results = {}       # {market: adb_local DataFrame} — manufacturer × period with optional MAT_Flag
mat_periods_results = {}  # {market: _mat_periods_adb DataFrame} — MAT TY/LY/2LY period lookup
market_metadata = {}   # {market: dict of computed metadata (category, descs, filters)}

for _mkt_params in markets_to_process:
    Local_Market = _mkt_params['Local_Market']
    min_period = _mkt_params['min_period']
    max_period = _mkt_params['max_period']

    _cfg             = market_config[Local_Market]
    _exc             = exceptions_config[Local_Market]
    mat_database     = _cfg['mat_database']

    print(f"Processing Market: {Local_Market}")
    print(f"Columns: mfr={_cfg['manufacturer_col']}  value={_cfg['value_col']}  volume={_cfg['volume_col']}")
    print(f"Exceptions: period_format={_exc['period_format']}, add_mat={_exc['add_mat_flag']}, mfr_comp={_exc['mfr_comparison_type']}")

    # Resolve valid market descriptions from dimmarket — these become the IN (...) filter
    # for all subsequent ADB queries. Ensures only recognized descriptions are included.
    _all_adb_descs = [r[0] for r in spark.sql(f"""
        SELECT DISTINCT Local_Market_Description
        FROM conformed.dimmarket gm
        WHERE Local_Market = '{Local_Market}'
        AND gm.Global_Total_Mkt_Flag='Y'
        ORDER BY Local_Market_Description
    """).collect() if r[0]]
    if not _all_adb_descs:
        print(f"  WARNING: No market descriptions found for Local_Market='{local_market}'. Skipping.")
        continue

    local_category_col = Filters_config[Local_Market]['Local_category_col']

    # Build SQL sub-category filter clause (e.g. "AND p.Local_Sub_Category IN ('RTE CEREALS','OATS')")
    # Empty if market has no sub-category constraint (e.g. Italy uses exclude logic instead)
    _sub_cats = Filters_config[Local_Market].get('Local_Sub_Category_col', [])
    if _sub_cats:
        _sc_in = ','.join(f"'{s.replace(chr(39), chr(39)*2)}'" for s in _sub_cats)
        _sub_cat_filter = f"AND p.Local_Sub_Category IN ({_sc_in})"
    else:
        _sub_cat_filter = ""

    # Period format: Germany uses ISO week format (YYYY_W##) for period grouping;
    # all other markets use standard date strings (yyyy-MM-dd)
    if _exc['period_format'] == 'week':
        _adb_period_select = "concat(year(t.Local_Time_Period), '_W', lpad(weekofyear(t.Local_Time_Period), 2, '0'))"
    else:
        _adb_period_select = "date_format(t.Local_Time_Period, 'yyyy-MM-dd')"

    # Discover min period override — allows Circana data to start from a different date
    # than ADB (e.g. when Circana history is shorter)
    _disc_min_period = _exc['disc_min_period_override'] or min_period

    # Product exclusion: SQL fragment to exclude specific product_ids from ADB
    # (e.g. UK excludes product_id 'H00190058@H01_L009')
    _product_exclusion = _exc['product_exclusion'] or ""

    # Use Local_Manufacturer for Germany (matches Circana's HERSTELLER names),
    # Global_Manufacturer for other markets
    _mfr_select = _exc.get('mfr_col', 'p.Global_Manufacturer')

    # Main ADB query: aggregates Value_000 and Volume_000 by Manufacturer × Period
    # Applies all market-specific filters: category, sub-category, product exclusion,
    # market descriptions, date range, and Global_Total_Mkt_Flag = 'Y'
    query = f"""
    SELECT
        {_mfr_select} AS Local_Manufacturer,
        {_adb_period_select} AS Local_Time_Period,
        SUM(f.Value_000) AS Value_000,
        SUM(f.Volume_000) AS Volume_000
    FROM conformed.factretailsales AS f
    JOIN conformed.dimproduct p
        ON f.LocalProductKey = p.LocalProductKey
    JOIN conformed.dimmarket gm
        ON f.LocalMarketKey = gm.LocalMarketKey
    JOIN conformed.dimperiod t
        ON f.LocalPeriodKey = t.LocalPeriodKey
    WHERE gm.Local_Market = '{Local_Market}'
      AND gm.Local_Market_Description IN ({', '.join(f"'{desc}'" for desc in _all_adb_descs)})
      AND t.Local_Time_Period >= '{min_period}'
      AND t.Local_Time_Period <= '{max_period}'
      AND gm.Global_Total_Mkt_Flag = 'Y'
      AND p.Local_Category IN ('{local_category_col}')
      {_sub_cat_filter}
      {_product_exclusion}
      AND p.Local_Manufacturer IS NOT NULL
    GROUP BY {_mfr_select}, {_adb_period_select}
    ORDER BY {_mfr_select}
    """

    adb_local = spark.sql(query)

    # MAT Flag join: period_mat_ytd maps each period date to MAT TY/LY/2LY flags.
    # Only added when add_mat_flag=True (AU, NZ, Italy, Germany). UK skips this because
    # its period comparison uses category_level logic instead of MAT-based filtering.
    # Period format must match the market's format: 'week' → YYYY_W## , 'date' → yyyy-MM-dd
    if _exc['period_format'] == 'week':
        _mat_period_expr = concat(
            date_format(col('Local_Time_Period'), 'yyyy'),
            lit('_W'),
            lpad(weekofyear(col('Local_Time_Period')).cast('string'), 2, '0')
        )
    else:
        _mat_period_expr = date_format(col('Local_Time_Period'), 'yyyy-MM-dd')

    _mat_periods_adb = (
        spark.table(f'adhoc.period_mat_ytd')
        .filter(col('Local_Period_Type') == 'Weekly')
        .filter(col('Database') == mat_database)
        .select(
            _mat_period_expr.alias('Local_Time_Period'),
            col('mat_flag').alias('MAT_Flag')
        )
        .distinct()
    )
    if _exc['add_mat_flag']:
        adb_local = adb_local.join(_mat_periods_adb, 'Local_Time_Period', 'left')
        print(f"  MAT_Flag added to adb_local (Database: {mat_database})")
    else:
        print(f"  {Local_Market}: MAT_Flag skipped (add_mat_flag=False)")

    # Store results for downstream comparison steps
    adb_results[Local_Market] = adb_local
    mat_periods_results[Local_Market] = _mat_periods_adb
    market_metadata[Local_Market] = {
        'local_category_col': local_category_col,
        '_disc_min_period': _disc_min_period,
        '_all_adb_descs': _all_adb_descs,
        '_sub_cat_filter': _sub_cat_filter,
    }
    print(f"  ADB query complete for {Local_Market}")
    display(adb_local)

Processing Market: Germany
Columns: mfr=HERSTELLER_TWH1 [ HERSTELLER_TWH1 ]  value=Verkauf Euro  volume=Verkauf Menge
Exceptions: period_format=week, add_mat=True, mfr_comp=all_periods
  MAT_Flag added to adb_local (Database: DE_R_NE3986)
  ADB query complete for Germany


Local_Time_Period,Local_Manufacturer,Value_000,Volume_000,MAT_Flag
2026_W19,STENGEL,0.0000,0.0000,MAT TY
2025_W48,BILLY TIGER,10.9345,0.3335,MAT TY
2026_W06,PWC ODRA,0.0289,0.0018,MAT TY
2023_W18,HIBA NATURAL FOOD,0.0000,0.0000,null
2024_W11,OLIMP NUTRITION,0.0106,0.0003,MAT 2LY
2025_W52,GEPA,0.0586,0.0024,MAT TY
2024_W23,NESTLE CPD,91.1678,6.5265,MAT 2LY
2025_W24,BESSERSNACKEN,0.0462,0.0008,MAT LY
2026_W11,BASIC,0.0000,0.0000,MAT TY
2024_W12,TEMPO SEKERLEME,0.0000,0.0000,MAT 2LY


# Step 3: Unify — Load Files for All Markets

In [0]:
%pip install -q openpyxl

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# ============================================================
# STEP 2: LOAD CIRCANA (DISCOVER) FILES & BUILD discover_agg
# Reads market-specific Circana Excel/CSV files, normalizes
# period formats, cleans numeric columns, and aggregates
# Value_000 / Volume_000 by Manufacturer × Period.
# ============================================================

import glob as _glob
import re as _re_mod
from datetime import datetime as _dt

# Base directory containing all market Circana input files
_input_base_dir = "/Workspace/Users/abhishek.sawlikar@genmills.com/Discover_vs_ADB_Check/Input File/Circana/Bars"

# Pre-compute known column headers for Excel header-row detection (avoids recomputing in loop).
# Includes generic headers ('Periods', 'Markets') plus all market-specific column names
# (manufacturer, value, volume) so the detector works regardless of which market file is opened.
_ALL_KNOWN_HEADERS = {'Periods', 'Markets'} | {
    v for cfg in market_config.values()
    for v in [cfg['manufacturer_col'], cfg['value_col'], cfg['volume_col']]
}

def _detect_excel_header_row(file_path_or_buf, nrows=20):
    """Detect the header row in an Excel file by finding the first row containing known column names.
    
    Circana files sometimes have title/metadata rows above the actual column headers.
    This scans the first `nrows` rows and returns the index of the first row that
    contains at least one recognized column name. Falls back to row 0 if none found.
    """
    import pandas as pd
    _probe_raw = pd.read_excel(file_path_or_buf, header=None, nrows=nrows)
    _header_row = next(
        (_i for _i, _row in _probe_raw.iterrows()
         if any(str(v).strip() in _ALL_KNOWN_HEADERS for v in _row.values if pd.notna(v))),
        0
    )
    return _header_row

def _parse_period_to_ddMMyy(val):
    """Convert diverse market-specific period strings to a uniform dd/MM/yy format.
    
    Handles four formats:
      A) DD-MM-YY or DD/MM/YY — Australia, New Zealand
      B) '<N> w/e DD Mon, YY' — UK (week-ending date)
      C) 'W <N> YYYY' — Germany (ISO week number)
      D) 'Settimana <N>, YYYY' or 'S <N> YYYY' — Italy (Italian week number)
    
    Returns empty string if parsing fails (filtered out downstream).
    """
    s = str(val).strip()
    if not s or s == 'nan':
        return ''
    # Format A: contains DD-MM-YY or DD/MM/YY (AU, NZ)
    m = _re_mod.search(r'(\d{2})[/-](\d{2})[/-](\d{2,4})$', s)
    if m:
        dd, mm, yy = m.group(1), m.group(2), m.group(3)[-2:]
        return f"{dd}/{mm}/{yy}"
    # Format B: '<N> w/e DD Mon, YY' (UK)
    m = _re_mod.search(r'w/e\s+(\d{1,2})\s+([A-Za-z]+),?\s+(\d{2,4})', s)
    if m:
        try:
            day, mon, yr = m.group(1), m.group(2), m.group(3)[-2:]
            dt = _dt.strptime(f"{day} {mon} {yr}", "%d %b %y")
            return dt.strftime("%d/%m/%y")
        except ValueError:
            return ''
    # Format C: 'W <N> YYYY' (Germany) — use ISO 8601 week calendar
    m = _re_mod.match(r'W\s*(\d+)\s+(\d{4})', s)
    if m:
        try:
            wk, yr = int(m.group(1)), int(m.group(2))
            dt = _dt.fromisocalendar(yr, wk, 7)  # Sunday (day 7) of ISO week
            return dt.strftime("%d/%m/%y")
        except ValueError:
            return ''
    # Format D: 'Settimana <N>, YYYY' or 'S <N> YYYY' (Italy)
    m = _re_mod.match(r'(?:Settimana|S)\s*(\d+),?\s*(\d{4})', s, _re_mod.IGNORECASE)
    if m:
        try:
            wk, yr = int(m.group(1)), int(m.group(2))
            dt = _dt.strptime(f"{yr} {wk} 0", "%Y %W %w")
            return dt.strftime("%d/%m/%y")
        except ValueError:
            return ''
    return ''


# --- Accumulators: store per-market Discover results for downstream comparisons ---
discover_results = {}      # {market: discover_agg DataFrame} — with MAT_Flag join (for mfr comparison)
discover_results_all = {}  # {market: discover_agg_all DataFrame} — all periods, no MAT filter (for period comparison)

for _mkt_params in markets_to_process:
    Local_Market = _mkt_params['Local_Market']
    min_period = _mkt_params['min_period']
    max_period = _mkt_params['max_period']

    if Local_Market not in market_metadata:
        continue  # Skip if Step 1 failed for this market

    _cfg = market_config[Local_Market]
    _exc = exceptions_config[Local_Market]
    manufacturer_col = _cfg['manufacturer_col']
    value_col = _cfg['value_col']
    volume_col = _cfg['volume_col']
    mat_database = _cfg['mat_database']
    _disc_min_period = market_metadata[Local_Market]['_disc_min_period']
    print(f"Loading Discover for: {Local_Market}")
    
    # File resolution: either use an explicit filename from exceptions_config,
    # or auto-detect via glob pattern matching on the input directory
    _circana_file_cfg = _exc.get('circana_file')
    _circana_sheet = _exc.get('circana_sheet')

    if _circana_file_cfg:
        # Explicit file specified in Filters_config
        file_path = os.path.join(_input_base_dir, _circana_file_cfg)
        file_type = 'csv' if _circana_file_cfg.lower().endswith('.csv') else 'excel'
        print(f"  Loading Discover file (explicit): {file_path}")
        if not os.path.exists(file_path):
            print(f"  WARNING: Explicit Discover file not found: {file_path}. Skipping.")
            continue
        if file_type == 'csv':
            pdf = pd.read_csv(file_path)
        elif _circana_sheet:
            try:
                pdf = pd.read_excel(file_path, sheet_name=_circana_sheet)
                print(f"  Sheet: {_circana_sheet}  |  Columns: {list(pdf.columns)}")
            except ValueError:
                import openpyxl
                _wb = openpyxl.load_workbook(file_path, read_only=True)
                _available_sheets = _wb.sheetnames
                _wb.close()
                print(f"  WARNING: Sheet '{_circana_sheet}' not found in {os.path.basename(file_path)}. Available: {_available_sheets}. Skipping market.")
                continue
        else:
            _header_row = _detect_excel_header_row(file_path)
            pdf = pd.read_excel(file_path, header=_header_row)
    else:
        # Auto-detect Discover file via glob pattern
        _primary  = _glob.glob(f"{_input_base_dir}/{Local_Market}_[Cc]ircana*.xlsx") or \
                    _glob.glob(f"{_input_base_dir}/{Local_Market}_[Cc]ircana*.csv")
        _fallback = (_glob.glob(f"{_input_base_dir}/{Local_Market}*.xlsx") or
                     _glob.glob(f"{_input_base_dir}/{Local_Market}*.csv")) if not _primary else []
        _found    = _primary or _fallback
        if not _found:
            print(f"  WARNING: No Circana file found for '{Local_Market}'. Skipping.")
            continue
        _detected_file = os.path.basename(_found[0])
        file_path = os.path.join(_input_base_dir, _detected_file)
        file_type = 'csv' if _detected_file.lower().endswith('.csv') else 'excel'
        print(f"  Auto-detected file: {file_path}")
        if file_type == 'csv':
            pdf = pd.read_csv(file_path)
        else:
            _header_row = _detect_excel_header_row(file_path)
            pdf = pd.read_excel(file_path, header=_header_row)

    # Detect the period/time column (varies by market file: 'Periods', 'Time', or 'Period')
    # and normalize all date strings to dd/MM/yy via _parse_period_to_ddMMyy
    _period_col = next((c for c in ['Periods', 'Time', 'Period'] if c in pdf.columns), None)
    if _period_col is None:
        print(f"  WARNING: No time/period column found. Columns: {list(pdf.columns)}. Skipping.")
        continue
    pdf[_period_col] = pdf[_period_col].apply(_parse_period_to_ddMMyy)

    # Coerce mixed-type object columns to string for Arrow/Spark compatibility.
    # Without this, pandas DataFrames with mixed int/str columns fail on createDataFrame.
    # Preserves NaN as actual null (not the string 'nan').
    _obj_cols = pdf.select_dtypes(include=['object']).columns
    _null_mask = pdf[_obj_cols].isna()
    pdf[_obj_cols] = pdf[_obj_cols].astype(str).where(~_null_mask)

    # Build df_cleansed: convert pandas → Spark, strip non-numeric chars from value/volume
    # columns (handles currency symbols, thousands separators), and filter null manufacturers
    _sdf = spark.createDataFrame(pdf)
    _new_cols = {
        "Local_Period": col(f"`{_period_col}`"),
        value_col: regexp_replace(col(f"`{value_col}`").cast('string'), r"[^0-9.eE+\-]", ""),
        volume_col: regexp_replace(col(f"`{volume_col}`").cast('string'), r"[^0-9.eE+\-]", ""),
    }
    df_cleansed = _sdf.withColumns(_new_cols).filter(col(f"`{manufacturer_col}`").isNotNull())


    # Period expression: must match the format used in ADB (Step 1) for correct join.
    # 'week' → YYYY_W## (Germany), 'date' → yyyy-MM-dd (all others)
    if _exc['period_format'] == 'week':
        _discover_period_expr = concat(
            date_format(to_date(col('Local_Period'), 'dd/MM/yy'), 'yyyy'),
            lit('_W'),
            lpad(weekofyear(to_date(col('Local_Period'), 'dd/MM/yy')).cast('string'), 2, '0')
        )
    else:
        _discover_period_expr = date_format(to_date(col('Local_Period'), 'dd/MM/yy'), 'yyyy-MM-dd')

    # Apply Discover-side sub-segment exclusion (Italy only: excludes 'OATS' from 'Tipo Valori')
    # This is the Discover counterpart to the ADB sub-category filter in Step 1
    # _exclude_tipo = Filters_config[detected_market].get('Discover_exclude_tipo', [])
    # if _exclude_tipo:
    #      print(f"  Excluding Discover 'Tipo Valori' segments: {_exclude_tipo}")
    #      df_cleansed = df_cleansed.filter(~col('`Tipo Valori`').isin(_exclude_tipo))

    # Some Circana files include their own MAT flag column (e.g. AU/NZ).
    # If present, use it directly instead of joining period_mat_ytd.
    _discover_mat_col = Filters_config[Local_Market].get('discover_mat_col')
    _disc_group_cols = ["Local_Manufacturer", "Local_Time_Period"]

    # Build discover_agg: parse dates, cast value/volume to double, and aggregate.
    # Uses single .withColumns() call for efficiency (avoids sequential .withColumn() overhead)
    _discover_cols = {
        '_date': to_date(col('Local_Period'), 'dd/MM/yy'),
        'Local_Time_Period': _discover_period_expr,
        'Value_000': expr(f"coalesce(try_cast(`{value_col}` as double), 0)"),
        'Volume_000': expr(f"coalesce(try_cast(`{volume_col}` as double), 0)"),
    }
    if _discover_mat_col:
        _discover_cols['MAT_Flag'] = col(f'`{_discover_mat_col}`')
        _disc_group_cols.append('MAT_Flag')
        print(f"  Using Discover file's own MAT column: '{_discover_mat_col}'")

    _discover_base = (
        df_cleansed
        .filter(col("Local_Period") != "")
        .withColumnRenamed(manufacturer_col, "Local_Manufacturer")
        .withColumns(_discover_cols)
        .filter(
            col('_date').isNotNull()
            & col('_date').between(_disc_min_period, max_period)
            & col('Local_Time_Period').isNotNull()
            & col('Local_Manufacturer').isNotNull()
        )
    )

    discover_agg = (
        _discover_base
        .groupBy(*_disc_group_cols)
        .agg(
            spark_sum("Value_000").alias("Value_000"),
            spark_sum("Volume_000").alias("Volume_000")
        )
    )

    # Save full aggregation (all periods, no MAT filter) for Step 4 period comparison
    discover_agg_all = discover_agg

    # Add MAT_Flag via period_mat_ytd join (inner join limits to MAT TY/LY/2LY periods).
    # Only applied when: (1) file doesn't have its own MAT column, and (2) add_mat_flag=True.
    # The MAT-filtered version is used in Step 3 manufacturer comparison.
    if not _discover_mat_col:
        # Period format must match: 'week' → YYYY_W## , 'date' → yyyy-MM-dd
        if _exc['period_format'] == 'week':
            _mat_period_expr_disc = concat(
                date_format(col('Local_Time_Period'), 'yyyy'),
                lit('_W'),
                lpad(weekofyear(col('Local_Time_Period')).cast('string'), 2, '0')
            )
        else:
            _mat_period_expr_disc = date_format(col('Local_Time_Period'), 'yyyy-MM-dd')

        _mat_periods_disc = (
            spark.table('glbl_cpw_prod.adhoc.period_mat_ytd')
            .filter(col('Local_Period_Type') == 'Weekly')
            .filter(col('Database') == mat_database)
            .select(
                _mat_period_expr_disc.alias('Local_Time_Period'),
                col('mat_flag').alias('MAT_Flag')
            )
            .distinct()
        )
        if _exc['add_mat_flag']:
            discover_agg = discover_agg.join(_mat_periods_disc, 'Local_Time_Period', 'inner')
            print(f"  MAT_Flag added to discover_agg (Database: {mat_database})")
        else:
            print(f"  {Local_Market}: MAT_Flag skipped for discover_agg (add_mat_flag=False)")
    else:
        print(f"  MAT_Flag sourced from Discover file column '{_discover_mat_col}'")

    discover_results[Local_Market] = discover_agg
    discover_results_all[Local_Market] = discover_agg_all
    print(f"  Discover load complete for {Local_Market}")
    display(discover_agg)

Loading Discover for: Germany
  Auto-detected file: /Workspace/Users/abhishek.sawlikar@genmills.com/Discover_vs_ADB_Check/Input File/Circana/Bars/Germany_Circana_270826.xlsx
  MAT_Flag added to discover_agg (Database: DE_R_NE3986)
  Discover load complete for Germany


Local_Time_Period,Local_Manufacturer,Value_000,Volume_000,MAT_Flag
2023_W04,ALL STARS FITNESS PRODUCTS,118458.4002822639,3368.365310896183,null
2024_W01,ALL STARS FITNESS PRODUCTS,100230.47249056364,2839.169037818168,MAT 2LY
2024_W08,ALL STARS FITNESS PRODUCTS,161771.0732989431,4512.209536252088,MAT 2LY
2024_W25,ALL STARS FITNESS PRODUCTS,213808.04739499,5864.147081565816,MAT 2LY
2025_W05,ALL STARS FITNESS PRODUCTS,224052.1830645696,6033.524508228527,MAT LY
2025_W16,ALL STARS FITNESS PRODUCTS,193983.5977818881,5304.859216843692,MAT LY
2026_W07,ALL STARS FITNESS PRODUCTS,181531.1273681891,5307.9883380295105,MAT TY
2023_W44,DR. SCHAER,3375.6778365752225,79.60131765959447,MAT 2LY
2023_W51,DR. SCHAER,3165.339387545681,75.16615491020981,MAT 2LY
2024_W15,DR. SCHAER,3778.5104451423513,89.53523077581072,MAT 2LY


# Step 4: Manufacturer Level ADB vs Unify Comparison — All Markets

In [0]:
# ============================================================
# STEP 4: MANUFACTURER-LEVEL Unify vs ADB COMPARISON - ALL MARKETS
# Comparison type driven by exceptions_config['mfr_comparison_type']
# Layout: ADB | Unify (Discover/1000) | Diff. ADB vs Unify
# ============================================================

for _mkt_params in markets_to_process:
    Local_Market = _mkt_params['Local_Market']
    if Local_Market not in discover_results or Local_Market not in adb_results:
        print(f"\nSkipping {Local_Market} - missing data from prior steps.")
        continue

    _cfg = market_config[Local_Market]
    _exc = exceptions_config[Local_Market]
    value_col = _cfg['value_col']
    volume_col = _cfg['volume_col']
    _meta = market_metadata[Local_Market]
    local_category_col = _meta['local_category_col']
    _all_adb_descs = _meta['_all_adb_descs']
    adb_local = adb_results[Local_Market]
    discover_agg = discover_results[Local_Market]
    _mat_periods_adb = mat_periods_results[Local_Market]
    _mfr_comp_type = _exc['mfr_comparison_type']

    print(f"Manufacturer Comparison: {Local_Market} (type={_mfr_comp_type})")

    if _mfr_comp_type == 'category_level':
        # Category-level (UK): queries ADB at the full category level WITHOUT sub-category filter.
        # Restricted to MAT TY periods only for time alignment with Discover.
        # Also applies product exclusion and manufacturer name transforms (strip prefix, rename).
        _descs_sql_mfr = ', '.join(f"'{d}'" for d in _all_adb_descs)
        # Collect the MAT TY period dates to use as an IN (...) filter in the SQL query
        _mat_ty_periods = _mat_periods_adb.filter(col('MAT_Flag') == 'MAT TY').select('Local_Time_Period')
        _mat_ty_list = [r['Local_Time_Period'] for r in _mat_ty_periods.collect()]
        _mat_ty_in = ', '.join(f"'{p}'" for p in _mat_ty_list)
        _product_exclusion = _exc['product_exclusion'] or ""

        _mfr_adb_query = f"""
        SELECT
            p.Local_Manufacturer AS Local_Manufacturer,
            SUM(f.Value_000) AS ADB_Value,
            SUM(f.Volume_000) AS ADB_Volume
        FROM conformed.factretailsales AS f
        JOIN conformed.dimproduct p ON f.LocalProductKey = p.LocalProductKey
        JOIN conformed.dimmarket gm ON f.LocalMarketKey = gm.LocalMarketKey
        JOIN conformed.dimperiod t ON f.LocalPeriodKey = t.LocalPeriodKey
        WHERE gm.Local_Market = '{Local_Market}'
          AND gm.Local_Market_Description IN ({_descs_sql_mfr})
          AND gm.Global_Total_Mkt_Flag = 'Y'
          AND p.Local_Category IN ('{local_category_col}')
          {_product_exclusion}
          AND p.Local_Manufacturer IS NOT NULL
          AND p.Local_Manufacturer != 'Unknown'
          AND date_format(t.Local_Time_Period, 'yyyy-MM-dd') IN ({_mat_ty_in})
        GROUP BY p.Local_Manufacturer
        """
        _adb_mfr_agg = spark.sql(_mfr_adb_query)

        # Apply manufacturer name transforms to align ADB naming with Circana.
        # E.g. UK strips 'CEREALS_' prefix and renames 'NATURES PATH' → 'NATURES PATH FOODS'
        # Applied sequentially: strip_prefix first, then rename overrides.
        if _exc['mfr_name_transforms']:
            _mfr_expr = col('Local_Manufacturer')
            for _transform in _exc['mfr_name_transforms']:
                if _transform['type'] == 'strip_prefix':
                    _mfr_expr = regexp_replace(_mfr_expr, _transform['pattern'], '')
                elif _transform['type'] == 'rename':
                    _mfr_expr = when(_mfr_expr == _transform['from'], lit(_transform['to'])).otherwise(_mfr_expr)
            _adb_mfr_agg = _adb_mfr_agg.withColumns({'Local_Manufacturer': _mfr_expr})

        # Re-aggregate after name transforms (some manufacturers may merge after rename)
        _adb_mfr_agg = (
            _adb_mfr_agg
            .groupBy('Local_Manufacturer')
            .agg(
                spark_round(spark_sum('ADB_Value'), 4).alias('ADB_Value'),
                spark_round(spark_sum('ADB_Volume'), 4).alias('ADB_Volume')
            )
        )

        # Discover side: filter to MAT TY periods only, then aggregate to manufacturer level
        _disc_base = discover_agg.join(_mat_ty_periods, 'Local_Time_Period', 'inner')
        _disc_mfr_agg = (
            _disc_base
            .groupBy('Local_Manufacturer')
            .agg(
                spark_round(spark_sum('Value_000'), 4).alias('Discover_Value'),
                spark_round(spark_sum('Volume_000'), 4).alias('Discover_Volume')
            )
        )
        comparison_manufacturer = (
            _adb_mfr_agg.alias('adb')
            .join(_disc_mfr_agg.alias('disc'), ['Local_Manufacturer'], 'inner')
        )

    elif _mfr_comp_type == 'mat_join':
        # MAT join (Italy): inner join Discover and ADB on (Manufacturer, Period, MAT_Flag)
        # BEFORE aggregating to manufacturer. This ensures only matching period×MAT combinations
        # are compared, avoiding mismatches from missing periods on either side.
        _disc_base = discover_agg.filter(col('MAT_Flag').isin(_mat_flags))
        discover_mfr_total = (
            _disc_base
            .groupBy('Local_Manufacturer', 'Local_Time_Period', 'MAT_Flag')
            .agg(
                spark_round(spark_sum('Value_000'), 4).alias('Discover_Value'),
                spark_round(spark_sum('Volume_000'), 4).alias('Discover_Volume')
            )
        )
        _adb_base = adb_local.filter(col('MAT_Flag').isin(_mat_flags))
        adb_mfr_total = (
            _adb_base
            .groupBy('Local_Manufacturer', 'Local_Time_Period', 'MAT_Flag')
            .agg(
                spark_round(spark_sum('Value_000'), 4).alias('ADB_Value'),
                spark_round(spark_sum('Volume_000'), 4).alias('ADB_Volume')
            )
        )
        comparison_manufacturer = (
            discover_mfr_total.alias('disc')
            .join(adb_mfr_total.alias('adb'), ['Local_Manufacturer', 'Local_Time_Period', 'MAT_Flag'], 'inner')
            .groupBy('Local_Manufacturer')
            .agg(
                spark_round(spark_sum('ADB_Value'), 4).alias('ADB_Value'),
                spark_round(spark_sum('ADB_Volume'), 4).alias('ADB_Volume'),
                spark_round(spark_sum('Discover_Value'), 4).alias('Discover_Value'),
                spark_round(spark_sum('Discover_Volume'), 4).alias('Discover_Volume')
            )
        )

    elif _mfr_comp_type == 'all_periods':
        # All MAT periods (Germany): ADB queried via LocalPeriodKey join to period_mat_ytd
        # for precise MAT filtering (TY+LY+2LY combined). Discover side also filtered to
        # MAT flags. Both aggregated to manufacturer level, then inner joined.
        _descs_sql_mfr = ', '.join(f"'{d}'" for d in _all_adb_descs)
        _mfr_col_expr = _exc.get('mfr_col', 'p.Global_Manufacturer')
        _mat_database = _cfg['mat_database']
        _sub_cat_filter = _meta['_sub_cat_filter']

        _adb_mfr_agg = spark.sql(f"""
        SELECT
            {_mfr_col_expr} AS Local_Manufacturer,
            SUM(f.Value_000) AS ADB_Value,
            SUM(f.Volume_000) AS ADB_Volume
        FROM conformed.factretailsales AS f
        JOIN conformed.dimproduct p ON f.LocalProductKey = p.LocalProductKey
        JOIN conformed.dimmarket gm ON f.LocalMarketKey = gm.LocalMarketKey
        JOIN glbl_cpw_prod.adhoc.period_mat_ytd p_mat
            ON f.LocalPeriodKey = p_mat.LocalPeriodKey
            AND p_mat.Database = '{_mat_database}'
            AND p_mat.mat_flag IN ('MAT TY', 'MAT LY', 'MAT 2LY')
        WHERE gm.Local_Market = '{Local_Market}'
          AND gm.Local_Market_Description IN ({_descs_sql_mfr})
          AND gm.Global_Total_Mkt_Flag = 'Y'
          AND p.Local_Category = '{local_category_col}'
          {_sub_cat_filter}
          AND {_mfr_col_expr} IS NOT NULL
        GROUP BY {_mfr_col_expr}
        """)

        # Discover side: filter to MAT flags, aggregate to manufacturer level
        _disc_mfr_agg = (
            discover_agg
            .filter(col('MAT_Flag').isin(_mat_flags))
            .groupBy('Local_Manufacturer')
            .agg(
                spark_sum('Value_000').alias('Discover_Value'),
                spark_sum('Volume_000').alias('Discover_Volume')
            )
        )

        # Full outer join at manufacturer level — shows unmatched manufacturers
        # from either side (matches Local_Manufacturer_Pivot reference layout)
        comparison_manufacturer = (
            _adb_mfr_agg.alias('adb')
            .join(_disc_mfr_agg.alias('disc'), ['Local_Manufacturer'], 'full')
        )

    else:  # 'mat' — standard MAT filter approach (AU, NZ)
        # Default: filter both sides to MAT TY/LY/2LY flags, aggregate each to manufacturer
        # independently, then inner join. Simpler than mat_join but doesn't enforce
        # period-level alignment (trusts that both sides have the same MAT periods).
        _disc_base = discover_agg.filter(col('MAT_Flag').isin(_mat_flags))
        _adb_base = adb_local.filter(col('MAT_Flag').isin(_mat_flags))
        _adb_mfr_agg = (
            _adb_base
            .groupBy('Local_Manufacturer')
            .agg(
                spark_round(spark_sum('Value_000'), 4).alias('ADB_Value'),
                spark_round(spark_sum('Volume_000'), 4).alias('ADB_Volume')
            )
        )
        _disc_mfr_agg = (
            _disc_base
            .groupBy('Local_Manufacturer')
            .agg(
                spark_round(spark_sum('Value_000'), 4).alias('Discover_Value'),
                spark_round(spark_sum('Volume_000'), 4).alias('Discover_Volume')
            )
        )
        comparison_manufacturer = (
            _adb_mfr_agg.alias('adb')
            .join(_disc_mfr_agg.alias('disc'), ['Local_Manufacturer'], 'inner')
        )

    # Dynamic Discover aliases reflect the market's native column names
    # (e.g. 'Unify_Sum_of_Value Sales' for UK, 'Unify_Sum_of_Verkauf Euro' for Germany)
    _disc_val_alias = f'Unify_Sum_of_{value_col}'
    _disc_vol_alias = f'Unify_Sum_of_{volume_col}'

    # Final result layout: ADB | Unify (Discover ÷ 1000 to match ADB's _000 units) | Diff
    # Diff = ADB - Unify; positive means ADB is higher than Circana source
    result_manufacturer = comparison_manufacturer.select(
        col('Local_Manufacturer'),
        spark_round(col('ADB_Value'), 4).alias('Conformed_Value_000'),
        spark_round(col('ADB_Volume'), 4).alias('Conformed_Volume_000'),
        spark_round(col('Discover_Value') / 1000, 6).alias(_disc_val_alias),
        spark_round(col('Discover_Volume') / 1000, 6).alias(_disc_vol_alias),
        spark_round(
            coalesce(col('ADB_Value'), lit(0)) - coalesce(col('Discover_Value'), lit(0)) / 1000,
            6
        ).alias('Value_Diff'),
        spark_round(
            coalesce(col('ADB_Volume'), lit(0)) - coalesce(col('Discover_Volume'), lit(0)) / 1000,
            6
        ).alias('Volume_Diff')
    ).orderBy('Local_Manufacturer')

    # Append a Grand Total row summing all manufacturers for quick validation
    grand_total = result_manufacturer.select(
        lit('Grand Total').alias('Local_Manufacturer'),
        spark_round(spark_sum('Conformed_Value_000'), 4).alias('Conformed_Value_000'),
        spark_round(spark_sum('Conformed_Volume_000'), 4).alias('Conformed_Volume_000'),
        spark_round(spark_sum(col(_disc_val_alias)), 6).alias(_disc_val_alias),
        spark_round(spark_sum(col(_disc_vol_alias)), 6).alias(_disc_vol_alias),
        spark_round(spark_sum('Value_Diff'), 6).alias('Value_Diff'),
        spark_round(spark_sum('Volume_Diff'), 6).alias('Volume_Diff')
    )

    result_manufacturer_with_total = result_manufacturer.unionByName(grand_total)

    # Display summary header and final comparison table
    print(f'Manufacturer-level comparison for market : {Local_Market}')
    print(f'Category filter                    : Local_Category = {local_category_col!r}')
    print(f'Comparison type                    : {_mfr_comp_type}')
    if _mfr_comp_type == 'all_periods':
        print(f'Aggregation                        : All MAT periods (TY+LY+2LY via LocalPeriodKey)')
    else:
        print(f'MAT filter                         : {_mat_flags}')
    print('Layout: ADB | Unify | Diff. ADB vs Unify')
    display(result_manufacturer_with_total)

Manufacturer Comparison: Germany (type=all_periods)
Manufacturer-level comparison for market : Germany
Category filter                    : Local_Category = 'Bars'
Comparison type                    : all_periods
Aggregation                        : All MAT periods (TY+LY+2LY via LocalPeriodKey)
Layout: ADB | Unify | Diff. ADB vs Unify


Local_Manufacturer,Conformed_Value_000,Conformed_Volume_000,Unify_Sum_of_Verkauf Euro,Unify_Sum_of_Verkauf Menge,Value_Diff,Volume_Diff
3BEARS,2915.8291,88.2815,2915.827054,88.279668,0.002046,0.001832
ACTIVE NUTRITION,54035.4103,1631.1632,54035.411057,1631.166053,-7.57E-4,-0.002853
ADONIS SMART FOODS,273.5627,6.3645,273.557255,6.3644,0.005445,1.0E-4
AHEAD,9207.3207,183.8233,9207.319721,183.822191,9.79E-4,0.001109
ALL STARS FITNESS PRODUCTS,26907.6877,745.8324,26907.688412,745.830038,-7.12E-4,0.002362
ALLOS WALTER LANG,48.3983,1.4411,48.394988,1.442432,0.003312,-0.001332
ALNATURA,35645.8439,1532.9043,35645.841239,1532.904625,0.002661,-3.25E-4
ALNAVIT,428.1375,13.9014,428.135341,13.902039,0.002159,-6.39E-4
ALPHA REPUBLIC,1748.4682,43.1799,1748.46877,43.180017,-5.7E-4,-1.17E-4
ALTINDAG SUESSWAREN,0.2360,0.0111,0.23528,0.00965,7.2E-4,0.00145


# Step 5: Period-level ADB vs Unify Comparison — All Markets

In [0]:
# ============================================================
# STEP 5: PERIOD-LEVEL Unify vs ADB COMPARISON - ALL MARKETS
# Comparison type driven by exceptions_config['period_comparison_type']
# Layout: ADB | Unify (Discover/1000) | Diff. ADB vs Unify
# ============================================================

# Optional manufacturer filter for period-level drill-down.
# Set to a manufacturer name string (e.g. 'GENERAL MILLS') to see only that mfr's periods;
# leave as None to aggregate ALL manufacturers into period-level totals.
_period_mfr_filter = None

for _mkt_params in markets_to_process:
    Local_Market = _mkt_params['Local_Market']
    min_period = _mkt_params['min_period']
    max_period = _mkt_params['max_period']

    if Local_Market not in discover_results_all or Local_Market not in adb_results:
        print(f"\nSkipping {Local_Market} - missing data from prior steps.")
        continue

    _cfg = market_config[Local_Market]
    _exc = exceptions_config[Local_Market]
    value_col = _cfg['value_col']
    volume_col = _cfg['volume_col']
    _meta = market_metadata[Local_Market]
    local_category_col = _meta['local_category_col']
    _all_adb_descs = _meta['_all_adb_descs']
    adb_local_all = adb_results[Local_Market]
    discover_agg_all = discover_results_all[Local_Market]
    _mat_periods_adb = mat_periods_results[Local_Market]
    _period_comp_type = _exc['period_comparison_type']

    print(f"Period Comparison: {Local_Market} (type={_period_comp_type})")

    if _period_comp_type == 'mat_flags':
        # Category-level MAT-filtered comparison (matches Local_Category_Pivot reference):
        # ADB re-queried with MAT filter via period_mat_ytd join, grouped by period only
        # (no manufacturer matching). Discover also filtered to MAT flags, grouped by period.
        _descs_sql_period = ', '.join(f"'{d}'" for d in _all_adb_descs)
        _mat_database = _cfg['mat_database']
        _sub_cat_filter = _meta.get('_sub_cat_filter', '')

        _period_adb_query = f"""
        SELECT
            date_format(t.Local_Time_Period, 'yyyy-MM-dd') AS Local_Time_Period,
            SUM(f.Value_000) AS ADB_Value,
            SUM(f.Volume_000) AS ADB_Volume
        FROM conformed.factretailsales AS f
        JOIN conformed.dimproduct p ON f.LocalProductKey = p.LocalProductKey
        JOIN conformed.dimmarket gm ON f.LocalMarketKey = gm.LocalMarketKey
        JOIN conformed.dimperiod t ON f.LocalPeriodKey = t.LocalPeriodKey
        JOIN glbl_cpw_prod.adhoc.period_mat_ytd p_mat
            ON f.LocalPeriodKey = p_mat.LocalPeriodKey
            AND p_mat.Database = '{_mat_database}'
            AND p_mat.mat_flag IN ('MAT TY', 'MAT LY', 'MAT 2LY')
        WHERE gm.Local_Market = '{Local_Market}'
          AND gm.Local_Market_Description IN ({_descs_sql_period})
          AND gm.Global_Total_Mkt_Flag = 'Y'
          AND p.Local_Category = '{local_category_col}'
          {_sub_cat_filter}
        GROUP BY date_format(t.Local_Time_Period, 'yyyy-MM-dd')
        """
        adb_period_total = spark.sql(_period_adb_query)

        # Discover: filter to MAT flags, aggregate to period level directly
        _disc_period_base = discover_results[Local_Market].filter(col('MAT_Flag').isin(_mat_flags))
        discover_period_total = (
            _disc_period_base
            .groupBy('Local_Time_Period')
            .agg(
                spark_round(spark_sum('Value_000'), 4).alias('Discover_Value'),
                spark_round(spark_sum('Volume_000'), 4).alias('Discover_Volume')
            )
        )

        comparison_period = (
            adb_period_total.alias('adb')
            .join(discover_period_total.alias('disc'), ['Local_Time_Period'], 'inner')
        )
        _period_adb_note = 'Category-level MAT-filtered (via period_mat_ytd)'

    elif _period_comp_type == 'category_level':
        # Category-level (UK): ADB queried at full RTEC category (no sub-category filter)
        # with product exclusion. This gives the broadest ADB total per period.
        # On the Discover side, prefers pre-aggregated category totals from the
        # Circana 'Period Level' sheet (cols I-L), falling back to product-level rollup.
        _descs_sql_period = ', '.join(f"'{d}'" for d in _all_adb_descs)
        _product_exclusion = _exc['product_exclusion'] or ""
        _period_adb_query = f"""
        SELECT
            date_format(t.Local_Time_Period, 'yyyy-MM-dd') AS Local_Time_Period,
            SUM(f.Value_000) AS Value_000,
            SUM(f.Volume_000) AS Volume_000
        FROM conformed.factretailsales AS f
        JOIN conformed.dimproduct p ON f.LocalProductKey = p.LocalProductKey
        JOIN conformed.dimmarket gm ON f.LocalMarketKey = gm.LocalMarketKey
        JOIN conformed.dimperiod t ON f.LocalPeriodKey = t.LocalPeriodKey
        WHERE gm.Local_Market = '{Local_Market}'
          AND gm.Local_Market_Description IN ({_descs_sql_period})
          AND gm.Global_Total_Mkt_Flag = 'Y'
          AND p.Local_Category IN ('{local_category_col}')
          {_product_exclusion}
          AND t.Local_Time_Period >= '{min_period}'
          AND t.Local_Time_Period <= '{max_period}'
        GROUP BY date_format(t.Local_Time_Period, 'yyyy-MM-dd')
        """
        _adb_period_df = spark.sql(_period_adb_query)

        # Circana period totals: prefer Category-level totals from Period Level sheet.
        # The Period Level sheet (cols I–L) has pre-aggregated category-level totals
        # that include ALL products (even those not in the Unify File product list),
        # giving a true category total comparable to the ADB category-level query above.
        _disc_file_cfg = _exc.get('circana_file')
        _disc_file_path = os.path.join(_input_base_dir, _disc_file_cfg) if _disc_file_cfg else None
        if _disc_file_path and os.path.exists(_disc_file_path):
            _pe_right = (
                _pe_right
                .dropna(subset=['Local_Period_Date'])
                .loc[lambda df: ~df['Local_Period_Date'].astype(str).str.contains('Grand|MAT|Total', na=False)]
                .assign(
                    Local_Time_Period=lambda df: pd.to_datetime(df['Local_Period_Date'], errors='coerce', format='mixed').dt.strftime('%Y-%m-%d'),
                    Category_Value=lambda df: pd.to_numeric(df['Category_Value'], errors='coerce').fillna(0),
                    Category_Volume=lambda df: pd.to_numeric(df['Category_Volume'], errors='coerce').fillna(0)
                )
                .dropna(subset=['Local_Time_Period'])
            )
            _disc_period_pdf = _pe_right[['Local_Time_Period', 'Category_Value', 'Category_Volume']].rename(
                columns={'Category_Value': 'Discover_Value', 'Category_Volume': 'Discover_Volume'}
            )
            discover_period_total = spark.createDataFrame(_disc_period_pdf)
            print(f"  Discover period source: Category-level totals from Period Level sheet ({len(_disc_period_pdf)} periods)")
        else:
            # Fallback: when no Circana file is configured or file doesn't exist,
            # aggregate product-level data from the Unify File (discover_agg_all).
            # This may differ slightly from the Period Level sheet totals if
            # the Unify File doesn't include all category products.
            _disc_period_base = discover_agg_all
            if _period_mfr_filter:
                _disc_period_base = _disc_period_base.filter(col('Local_Manufacturer') == _period_mfr_filter)
            discover_period_total = (
                _disc_period_base
                .groupBy('Local_Time_Period')
                .agg(
                    spark_sum('Value_000').alias('Discover_Value'),
                    spark_sum('Volume_000').alias('Discover_Volume')
                )
            )
            print("  Discover period source: Product-level Unify File aggregation (fallback)")

        adb_period_total = (
            _adb_period_df
            .groupBy('Local_Time_Period')
            .agg(
                spark_round(spark_sum('Value_000'), 4).alias('ADB_Value'),
                spark_round(spark_sum('Volume_000'), 4).alias('ADB_Volume')
            )
        )

        # Inner join: only periods present in both ADB and Discover are compared
        comparison_period = (
            adb_period_total.alias('adb')
            .join(discover_period_total.alias('disc'), ['Local_Time_Period'], 'inner')
        )
        _period_adb_note = 'ADB at category level (no sub-category filter)'

    else:
        # All periods (Germany): MAT-filtered ADB via LocalPeriodKey join,
        # matched manufacturers between ADB and Discover, then rolled up to period level.
        # This ensures only manufacturers present in both sources contribute to totals.
        _descs_sql_period = ', '.join(f"'{d}'" for d in _all_adb_descs)
        _mfr_col_expr = _exc.get('mfr_col', 'p.Global_Manufacturer')
        _mat_database = _cfg['mat_database']
        _sub_cat_filter = _meta['_sub_cat_filter']

        # Period format expression for SQL
        if _exc['period_format'] == 'week':
            _period_sql_expr = "concat(year(t.Local_Time_Period), '_W', lpad(weekofyear(t.Local_Time_Period), 2, '0'))"
        else:
            _period_sql_expr = "date_format(t.Local_Time_Period, 'yyyy-MM-dd')"

        # Discover: filter to MAT flags
        _disc_period_base = discover_results[Local_Market].filter(col('MAT_Flag').isin(_mat_flags))

        if _period_mfr_filter:
            # Single-manufacturer drill-down
            _disc_period_base = _disc_period_base.filter(col('Local_Manufacturer') == _period_mfr_filter)
            _adb_period_filtered = adb_local_all.filter(col('Local_Manufacturer') == _period_mfr_filter)
            adb_period_total = (
                _adb_period_filtered
                .groupBy('Local_Time_Period')
                .agg(
                    spark_round(spark_sum('Value_000'), 4).alias('ADB_Value'),
                    spark_round(spark_sum('Volume_000'), 4).alias('ADB_Volume')
                )
            )
            _period_adb_note = f'All periods for manufacturer {_period_mfr_filter}'
        else:
            # ADB: re-query with MAT filter via LocalPeriodKey, grouped by mfr + period
            _period_adb_query = f"""
            SELECT
                {_mfr_col_expr} AS Local_Manufacturer,
                {_period_sql_expr} AS Local_Time_Period,
                SUM(f.Value_000) AS ADB_Value,
                SUM(f.Volume_000) AS ADB_Volume
            FROM conformed.factretailsales AS f
            JOIN conformed.dimproduct p ON f.LocalProductKey = p.LocalProductKey
            JOIN conformed.dimmarket gm ON f.LocalMarketKey = gm.LocalMarketKey
            JOIN conformed.dimperiod t ON f.LocalPeriodKey = t.LocalPeriodKey
            JOIN glbl_cpw_prod.adhoc.period_mat_ytd p_mat
                ON f.LocalPeriodKey = p_mat.LocalPeriodKey
                AND p_mat.Database = '{_mat_database}'
                AND p_mat.mat_flag IN ('MAT TY', 'MAT LY', 'MAT 2LY')
            WHERE gm.Local_Market = '{Local_Market}'
              AND gm.Local_Market_Description IN ({_descs_sql_period})
              AND gm.Global_Total_Mkt_Flag = 'Y'
              AND p.Local_Category = '{local_category_col}'
              {_sub_cat_filter}
              AND {_mfr_col_expr} IS NOT NULL
            GROUP BY {_mfr_col_expr}, {_period_sql_expr}
            """
            _adb_period_mfr_df = spark.sql(_period_adb_query)

            # Get matched manufacturers (present in both ADB and Discover)
            _adb_mfrs = _adb_period_mfr_df.select('Local_Manufacturer').distinct()
            _disc_mfrs = _disc_period_base.select('Local_Manufacturer').distinct()
            _matched_mfrs = _adb_mfrs.join(_disc_mfrs, 'Local_Manufacturer', 'inner')

            # Filter both sides to matched manufacturers, then roll up to period level
            adb_period_total = (
                _adb_period_mfr_df.join(_matched_mfrs, 'Local_Manufacturer', 'inner')
                .groupBy('Local_Time_Period')
                .agg(
                    spark_round(spark_sum('ADB_Value'), 4).alias('ADB_Value'),
                    spark_round(spark_sum('ADB_Volume'), 4).alias('ADB_Volume')
                )
            )
            _disc_period_base = _disc_period_base.join(_matched_mfrs, 'Local_Manufacturer', 'inner')
            _period_adb_note = 'MAT TY/LY/2LY via LocalPeriodKey (mfr-matched scope)'

        # Common: Discover period totals + join
        discover_period_total = (
            _disc_period_base
            .groupBy('Local_Time_Period')
            .agg(
                spark_round(spark_sum('Value_000'), 4).alias('Discover_Value'),
                spark_round(spark_sum('Volume_000'), 4).alias('Discover_Volume')
            )
        )

        comparison_period = (
            adb_period_total.alias('adb')
            .join(discover_period_total.alias('disc'), ['Local_Time_Period'], 'inner')
        )

    # --- Result formatting (common to all comparison types) ---

    _disc_val_alias = f'Unify_Sum_of_{value_col}'
    _disc_vol_alias = f'Unify_Sum_of_{volume_col}'

    # Sort key: Germany's YYYY_W## sorts lexicographically; others need date parse
    _sort_expr = col('Local_Time_Period') if Local_Market == 'Germany' else to_date(col('Local_Time_Period'), 'yyyy-MM-dd')

    # Final display: ADB | Unify (÷1000) | Diff (ADB - Unify)
    result_period = comparison_period.select(
        col('Local_Time_Period'),
        _sort_expr.alias('_sort_key'),
        spark_round(coalesce(col('ADB_Value'), lit(0)), 4).alias('Conformed_Value_000'),
        spark_round(coalesce(col('ADB_Volume'), lit(0)), 4).alias('Conformed_Volume_000'),
        spark_round(coalesce(col('Discover_Value'), lit(0)) / 1000, 6).alias(_disc_val_alias),
        spark_round(coalesce(col('Discover_Volume'), lit(0)) / 1000, 6).alias(_disc_vol_alias),
        spark_round(
            coalesce(col('ADB_Value'), lit(0)) - coalesce(col('Discover_Value'), lit(0)) / 1000,
            6
        ).alias('Value_Diff'),
        spark_round(
            coalesce(col('ADB_Volume'), lit(0)) - coalesce(col('Discover_Volume'), lit(0)) / 1000,
            6
        ).alias('Volume_Diff')
    ).orderBy('_sort_key').drop('_sort_key')

    # Add MAT TY summary row: filters result to only MAT TY periods and sums them.
    # Provides a quick validation point — the MAT TY total should match the
    # manufacturer-level Grand Total from Step 3 (when both use the same scope).
    _mat_ty_period_list = [r['Local_Time_Period'] for r in _mat_periods_adb.filter(col('MAT_Flag') == 'MAT TY').collect()]
    result_mat = result_period.filter(col('Local_Time_Period').isin(_mat_ty_period_list)).select(
        lit('MAT').alias('Local_Time_Period'),
        spark_round(spark_sum('Conformed_Value_000'), 4).alias('Conformed_Value_000'),
        spark_round(spark_sum('Conformed_Volume_000'), 4).alias('Conformed_Volume_000'),
        spark_round(spark_sum(col(_disc_val_alias)), 6).alias(_disc_val_alias),
        spark_round(spark_sum(col(_disc_vol_alias)), 6).alias(_disc_vol_alias),
        spark_round(spark_sum('Value_Diff'), 6).alias('Value_Diff'),
        spark_round(spark_sum('Volume_Diff'), 6).alias('Volume_Diff')
    )

    # Add Grand Total row: sums ALL periods (not just MAT TY) for full-range validation
    grand_total_period = result_period.select(
        lit('Grand Total').alias('Local_Time_Period'),
        spark_round(spark_sum('Conformed_Value_000'), 4).alias('Conformed_Value_000'),
        spark_round(spark_sum('Conformed_Volume_000'), 4).alias('Conformed_Volume_000'),
        spark_round(spark_sum(col(_disc_val_alias)), 6).alias(_disc_val_alias),
        spark_round(spark_sum(col(_disc_vol_alias)), 6).alias(_disc_vol_alias),
        spark_round(spark_sum('Value_Diff'), 6).alias('Value_Diff'),
        spark_round(spark_sum('Volume_Diff'), 6).alias('Volume_Diff')
    )

    # Stack: all individual periods + MAT TY summary + Grand Total
    result_period_with_total = result_period.unionByName(result_mat).unionByName(grand_total_period)

    # Display summary header and final period comparison table
    print(f'Period-level comparison for market : {Local_Market}')
    print(f'ADB scope                          : {_period_adb_note}')
    print(f'Category filter                    : Local_Category = {local_category_col!r}')
    print(f'Manufacturer filter                : {_period_mfr_filter or "All"}')
    print(f'Period range                       : {min_period} to {max_period}')
    print('Layout: ADB | Unify | Diff. ADB vs Unify')
    display(result_period_with_total)

Period Comparison: Germany (type=all_periods)
Period-level comparison for market : Germany
ADB scope                          : MAT TY/LY/2LY via LocalPeriodKey (mfr-matched scope)
Category filter                    : Local_Category = 'Bars'
Manufacturer filter                : All
Period range                       : 2023-01-08 to 2026-07-26
Layout: ADB | Unify | Diff. ADB vs Unify


Local_Time_Period,Conformed_Value_000,Conformed_Volume_000,Unify_Sum_of_Verkauf Euro,Unify_Sum_of_Verkauf Menge,Value_Diff,Volume_Diff
2023_W31,13361.4949,761.1480,13361.491169,761.144992,0.003731,0.003008
2023_W32,12763.3433,699.6414,12763.338866,699.638256,0.004434,0.003144
2023_W33,12150.1642,653.4461,12150.159424,653.445212,0.004776,8.88E-4
2023_W34,12212.4323,689.9026,12212.428265,689.901378,0.004035,0.001222
2023_W35,12920.3543,717.6097,12920.351951,717.608281,0.002349,0.001419
2023_W36,12939.8529,728.5301,12939.85009,728.529273,0.00281,8.27E-4
2023_W37,13129.0522,726.3106,13129.046061,726.309085,0.006139,0.001515
2023_W38,13536.1656,806.5041,13536.161212,806.501706,0.004388,0.002394
2023_W39,12802.3273,704.1532,12802.323953,704.151998,0.003347,0.001202
2023_W40,12066.9350,734.9199,12066.927173,734.919063,0.007827,8.37E-4
